<a href="https://colab.research.google.com/github/lidchen/ToyTransformer/blob/main/transformer_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset
import torch

ds = load_dataset("roneneldan/TinyStories")
train_ds = ds["train"]
text = ""
for i in range(5000):
    text += train_ds[i]["text"]
chars = sorted(list(set(text)))

stoi = {
    ch:i
    for i,ch in enumerate(chars)
}
itos = {
    i:ch
    for ch,i in stoi.items()
}
all_text = ""

for i in range(5000):
    all_text += train_ds[i]["text"]

data = torch.tensor(
  [stoi[c] for c in all_text],
  dtype=torch.long
)
def get_batch(B, T):
    ix = torch.randint(
        0,
        len(data) - T - 1,
        (B,)
    )
    x = torch.stack([
        data[i:i+T]
        for i in ix
    ])
    y = torch.stack([
        data[i+1:i+T+1]
        for i in ix
    ])
    return x, y

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class toyTransformer(nn.Module):
  def __init__(
    self,
    vocab_size,
    block_size, #T
    embed_dim, #C
    head_num, # H
    head_size # D
  ):
    super().__init__()
    self.block_size = block_size
    self.head_num = head_num
    self.head_size = head_size
    if (head_size * head_num) != embed_dim:
      print("Panic: head_size * head_num show equal to embed_dim")
      return
    self.token_embedding = nn.Embedding(
        vocab_size,
        embed_dim
    )
    self.position_embedding = nn.Embedding(
        block_size,
        embed_dim
    )
    # Multiple heads with fused projection
    self.query = nn.Linear(
        embed_dim,
        head_num * head_size, # H * D
        bias=False
    )
    self.key = nn.Linear(
        embed_dim,
        head_num * head_size,
        bias=False
    )
    self.value = nn.Linear(
        embed_dim,
        head_num * head_size,
        bias=False
    )
    self.lm_head = nn.Linear(
        embed_dim,
        vocab_size,
        bias=False
    )
    self.proj = nn.Linear(embed_dim, embed_dim)
    self.lm_head.weight = self.token_embedding.weight
    self.ln1 = nn.LayerNorm(embed_dim)
    self.ln2 = nn.LayerNorm(embed_dim)
    self.lnf = nn.LayerNorm(embed_dim)
    self.ffn = nn.Sequential(
      nn.Linear(embed_dim, 4 * embed_dim),
      nn.GELU(),
      nn.Linear(4 * embed_dim, embed_dim)
    )
    self.register_buffer(
      "tril",
      torch.tril(torch.ones(block_size, block_size))
    )

  def forward(self, idx, targets=None):
    B, T = idx.shape
    x = self.token_embedding(idx) + self.position_embedding(torch.arange(T, device=idx.device))

    # Pre-norm attention
    x_norm = self.ln1(x)

    q = self.query(x_norm)
    k = self.key(x_norm)
    v = self.value(x_norm)

    # resharp
    q = q.view(B, T, self.head_num, self.head_size)
    k = k.view(B, T, self.head_num, self.head_size)
    v = v.view(B, T, self.head_num, self.head_size)

    # group
    q = q.transpose(1, 2) # B T H D -> B H T D
    k = k.transpose(1, 2)
    v = v.transpose(1, 2)

    # attention weight
    weight = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)

    # mask
    weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

    # softmax
    weight = F.softmax(weight, dim=-1)

    out = weight @ v

    # reshape
    out = out.transpose(1, 2) # B H T D -> B T H D
    out = out.contiguous().view(B, T, self.head_num * self.head_size) # B T H D -> B T C

    # concat multiple heads
    x = self.proj(out)

    # Pre-norm FFN
    x = x + self.ffn(self.ln2(x))

    # final layerNorm
    x = self.lnf(x)

    # logits
    logits = self.lm_head(x)

    # loss
    loss = None
    if targets is not None:
      loss = F.cross_entropy(
          logits.view(-1, logits.shape[-1]),
          targets.view(-1)
      )
    return logits, loss


  @torch.no_grad()
  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, _ = self(idx)
      last_logits = logits[:, -1, :]
      probs = F.softmax(
        last_logits,
        dim=-1
      )
      next_idx = torch.multinomial(
        probs,
        num_samples=1
      )
      idx = torch.cat(
        [idx, next_idx],
        dim=1
      )
      idx = idx[:, -self.block_size:]
    return idx


In [ ]:
# train a new model
from google.colab import drive
drive.mount('/content/drive')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = toyTransformer(
  vocab_size=len(chars),
  block_size=128,
  embed_dim=256,
  head_num=8,
  head_size=32,
)
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=3e-4
)

for step in range(5000):
  idx, targets = get_batch(64, model.block_size)
  idx = idx.to(device)
  targets = targets.to(device)
  logits, loss = model(idx, targets)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if step % 1000 == 0:
    print(loss.item())

# train a new model
torch.save(model.state_dict(), "/content/drive/MyDrive/model.pt")
# torch.save( model.state_dict(), "model.pt" )

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cuda
41.89570617675781
2.96431303024292
2.7684826850891113
2.4880194664001465
2.436022996902466
2.397944450378418
2.351566791534424
2.3501925468444824
2.305143356323242
2.3107075691223145


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Continue training
# device = "cuda" if torch.cuda.is_available() else "cpu"
# print(device)

# model = toyTransformer(
#   vocab_size=len(chars),
#   block_size=128,
#   embed_dim=256,
#   head_size=64,
#   head_num=4
# )

model.load_state_dict(
  torch.load(
    "/content/drive/MyDrive/model.pt",
    map_location=device
  )
)

model.train()
model = model.to(device)

optimizer = torch.optim.Adam(
  model.parameters(),
  lr=1e-3
)

for step in range(10000):
  idx, targets = get_batch(64, model.block_size)
  idx = idx.to(device)
  targets = targets.to(device)
  logits, loss = model(idx, targets)

  optimizer.zero_grad()
  loss.backward()
  optimizer.step()
  if step % 1000 == 0:
    print(loss.item())

# torch.save( model.state_dict(), "model.pt" )
torch.save(model.state_dict(), "/content/drive/MyDrive/model.pt")


0.9315577149391174
0.9546154737472534
0.9430835247039795
0.9382233619689941
0.9308685064315796
0.8835148811340332
0.9231258630752563
0.9378710985183716
0.8856080770492554
0.9258873462677002


In [ ]:
model.load_state_dict(
  torch.load(
    "/content/drive/MyDrive/model.pt",
    map_location=device
  )
)
print("model loaded")
model.eval()
start = torch.tensor([[stoi["T"]]])
start = start.to(device)
out = model.generate(
  start,
  max_new_tokens=30000
)
print(out)

decoded = ''.join(
  [itos[i.item()] for i in out[0]]
)
print(decoded)

model loaded
tensor([[ 8,  1, 30,  1, 67, 62, 59, 51,  1, 72, 62, 68,  1, 49, 59, 52, 70,  1,
         49, 48, 50, 58,  1, 67, 62,  1, 67, 55, 52,  1, 72, 62, 68, 65,  1, 55,
         62, 60, 52,  8,  3, 34, 62, 60,  2,  3,  0,  0, 33, 48, 67, 52, 65,  1,
         67, 55, 62, 68, 54, 55, 67,  1, 53, 62, 65,  1, 48,  1, 50, 62, 60, 52,
          1, 65, 68, 61, 61, 56, 61, 54,  1, 67, 62,  1, 66, 48, 61, 54,  1, 56,
         61, 67, 62,  1, 67, 55, 52,  1, 63, 48, 65, 58,  8,  1, 29, 52, 65,  1,
         53, 65, 62, 70, 61,  6,  1, 67, 55, 52, 72,  1, 51, 56, 66, 50, 62, 69,
         52, 65]], device='cuda:0')
. I told you blew back to the your home."Mom!"

Later thought for a come running to sang into the park. Her frown, they discover


In [ ]:
# test
# note, this version separate linear layer for each head.
# for fused projection have:
# q = nn.Linear(C, H*D)

import torch
import torch.nn as nn
vocab_size = 5
block_size = 4   #T
embed_dim = 3 #C
head_size = 2
t_emb = nn.Embedding(vocab_size, embed_dim)
p_emb = nn.Embedding(block_size, embed_dim)

idx = torch.randint(5, (2, 4))
B, T = idx.shape
token_emb = t_emb(idx)
print("token_emb:",token_emb.shape)
print(token_emb)
pos_emb = p_emb(torch.arange(T))
print("pos_emb:", pos_emb.shape)
print(pos_emb)
x = token_emb + pos_emb
print("x:", x.shape)
print(x)

query = nn.Linear(embed_dim, head_size, bias = False)
print("query:", query)
q = query(x)
print("q:", q.shape)
k = query(x)

k_t = k.transpose(-2, -1)
print("k_t:", k_t.shape)
wei = q @ k_t
print("wei:", wei.shape)

v = query(x)
att = wei @ v
print("att:", att.shape)

token_emb: torch.Size([2, 4, 3])
tensor([[[-0.4170,  1.2050, -0.5670],
         [-0.1398, -0.3341, -1.2764],
         [ 0.3923,  1.3114, -0.3517],
         [ 0.3923,  1.3114, -0.3517]],

        [[-0.4170,  1.2050, -0.5670],
         [-0.1398, -0.3341, -1.2764],
         [ 0.3923,  1.3114, -0.3517],
         [-0.4170,  1.2050, -0.5670]]], grad_fn=<EmbeddingBackward0>)
pos_emb: torch.Size([4, 3])
tensor([[-0.1228,  1.9076, -0.5162],
        [-0.1953,  0.4026, -0.9442],
        [-0.9989, -0.9163, -0.3207],
        [ 1.5049,  1.4423,  0.8539]], grad_fn=<EmbeddingBackward0>)
x: torch.Size([2, 4, 3])
tensor([[[-0.5397,  3.1127, -1.0833],
         [-0.3351,  0.0685, -2.2205],
         [-0.6066,  0.3951, -0.6724],
         [ 1.8972,  2.7537,  0.5021]],

        [[-0.5397,  3.1127, -1.0833],
         [-0.3351,  0.0685, -2.2205],
         [-0.6066,  0.3951, -0.6724],
         [ 1.0879,  2.6473,  0.2868]]], grad_fn=<AddBackward0>)
query: Linear(in_features=3, out_features=2, bias=False)
q: torch